# Breast Cancer Prediction
### Machine Learning Classification | Akinsola Emmanuel
### University of Ibadan | github.com/akinsola-data

## 1. Load Dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("Shape:", df.shape)
print("\nFirst 3 rows:")
print(df.head(3))
print("\nTarget distribution:")
print(df['target'].value_counts())
print("\nTarget meaning: 0 = malignant (cancer), 1 = benign (no cancer)")

## 2. Exploratory Data Analysis

In [ ]:
print("=== Dataset Info ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Features: {df.shape[1] - 1}")

print("\n=== Basic Statistics ===")
print(df.describe().round(2))

## 3. Missing Value Check

In [ ]:
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

## 4. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Features:         {X_train.shape[1]}")
print(f"\nClass balance in training set:")
print(y_train.value_counts())
print(f"\nClass balance in test set:")
print(y_test.value_counts())

## 5. Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import numpy as np

models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

print(f"{'Model':<25} {'Train Acc':>10} {'Test Acc':>10} {'CV Mean':>10} {'CV Std':>10} {'Overfit?':>10}")
print("-" * 75)

for name, model in models.items():
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))

    cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_mean   = cv_scores.mean()
    cv_std    = cv_scores.std()

    overfit = "YES" if (train_acc - test_acc) > 0.05 else "No"

    results[name] = {
        'model':     model,
        'train_acc': train_acc,
        'test_acc':  test_acc,
        'cv_mean':   cv_mean,
        'cv_std':    cv_std,
        'overfit':   overfit
    }

    print(f"{name:<25} {train_acc:>9.2%} {test_acc:>10.2%} {cv_mean:>10.2%} {cv_std:>10.2%} {overfit:>10}")

## 6. Cross-Validation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

model_names = list(results.keys())
cv_means    = [results[m]['cv_mean'] * 100   for m in model_names]
cv_stds     = [results[m]['cv_std']  * 100   for m in model_names]
test_accs   = [results[m]['test_acc'] * 100  for m in model_names]

x = range(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar([i - width/2 for i in x], test_accs, width,
               label='Test Accuracy', color='#5B8DB8', edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], cv_means,  width,
               label='CV Mean Accuracy', color='#6BBF8E', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10)

ax.set_title('Model Comparison — Test Accuracy vs Cross-Validation Accuracy',
             fontsize=13, pad=15)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(85, 102)
ax.set_xticks(list(x))
ax.set_xticklabels(model_names)
ax.legend()
plt.tight_layout()
plt.savefig('cv_comparison.png', dpi=150)
plt.show()

## 7. Overfitting Analysis

In [ ]:
print("── Overfitting Check ──────────────────────────")
for name, res in results.items():
    gap = (res['train_acc'] - res['test_acc']) * 100
    status = '⚠ Possible overfit' if abs(gap) > 5 else '✓ Stable'
    print(f"{name:<25} Train-Test Gap: {gap:+.2f}%   {status}")

## 8. Confusion Matrix — Best Model

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

best_model = results['Random Forest']['model']
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant (0)', 'Benign (1)'],
            yticklabels=['Malignant (0)', 'Benign (1)'])
plt.title('Random Forest — Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred,
      target_names=['Malignant (0)', 'Benign (1)']))

## 9. Feature Importance

In [ ]:
rf_model = results['Random Forest']['model']

importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(9, 6))
top10.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Random Forest — Top 10 Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("\n=== Top 10 Features ===")
for i, (feat, score) in enumerate(top10.items(), 1):
    print(f"{i:>2}. {feat:<35} {score:.4f}")

## 10. Final Summary

In [ ]:
print("=" * 60)
print("       BREAST CANCER PREDICTION — FINAL RESULTS")
print("=" * 60)

print(f"\n{'Model':<25} {'Test Acc':>10} {'CV Mean':>10} {'CV Std':>8} {'Overfit':>8}")
print("-" * 60)
for name, r in results.items():
    print(f"{name:<25} {r['test_acc']:>9.2%} {r['cv_mean']:>10.2%} {r['cv_std']:>8.2%} {r['overfit']:>8}")

print("\n" + "=" * 60)
print("BEST MODEL: Random Forest")
print(f"  Test Accuracy : {results['Random Forest']['test_acc']:.2%}")
print(f"  CV Mean       : {results['Random Forest']['cv_mean']:.2%}")
print(f"  CV Std        : {results['Random Forest']['cv_std']:.2%}")
print(f"  Overfitting   : {results['Random Forest']['overfit']}")
print("\nTOP 2 PREDICTIVE FEATURES:")
top2 = importances.sort_values(ascending=False).head(2)
for i, (feat, score) in enumerate(top2.items(), 1):
    print(f"  {i}. {feat} ({score:.4f})")
print("\nCHARTS SAVED:")
print("  - cv_comparison.png")
print("  - confusion_matrix.png")
print("  - feature_importance.png")
print("=" * 60)
print("\nNotebook complete. Ready to upload to GitHub.")